In [20]:
%pip install lastfm


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement lastfm (from versions: none)
ERROR: No matching distribution found for lastfm


In [21]:
# MusicMatch Database Integration for Last.fm Enhanced Genre Data
from database_helper import MusicMatchDB
from lastfm import GenreExtractor

# Initialize database connection
try:
    db = MusicMatchDB()
    print("✅ Database connection successful!")
    
    # Initialize genre extractor
    extractor = GenreExtractor()
    print("✅ Genre extractor initialized!")
    
except Exception as e:
    print(f"❌ Setup failed: {e}")
    db = None
    extractor = None

✅ Database connection successful!
✅ Genre extractor initialized!


In [22]:
import http
import requests

In [23]:
API_KEY = "3a13f138ccc8df77f51f39f2ab0b3116"

In [24]:
artist = "The Weeknd"
album = "Hurry Up Tomorrow"

url = "https://ws.audioscrobbler.com/2.0"

params = {
    "method": "album.gettoptags",
    "artist": artist,
    "album": album,
    "api_key": API_KEY,
    "format": "json"
}

response = requests.get(url, params=params)

data = response.json()
tag_list = data['toptags']['tag']
artist_tag = {}
taglist = []

for tag in tag_list:
    taglist.append(tag['name'])

artist_tag[album] = taglist
print(artist_tag)


{'Hurry Up Tomorrow': ['2025', 'synthwave', 'alternative rnb', 'synthpop', 'alt-pop', 'pop', 'electropop', 'rnb', 'XOTWOD', 'electronic']}


In [25]:
from lastfm import GenreExtractor

extractor = GenreExtractor()
retrieved_tags = taglist
genres = extractor.extract_genres(retrieved_tags)

print("Filtered genre: ", genres)

Filtered genre:  ['synthwave', 'alternative rock', 'synthpop', 'alt-pop', 'pop', 'electropop', 'rnb', 'electronic']


In [26]:
# Enhanced Genre Processing and Database Integration
if db is not None and extractor is not None:
    try:
        # Process the album/artist tags we retrieved
        enhanced_genres = extractor.extract_genres(retrieved_tags)
        
        print(f"📊 Original tags count: {len(retrieved_tags)}")
        print(f"🎵 Filtered genres count: {len(enhanced_genres)}")
        print(f"✨ Enhanced genres: {enhanced_genres}")
        
        # If we want to save enhanced genre data for an artist/album
        # We could update our database with this enriched genre information
        
        # Example: Update artist with enhanced genres
        # artist_result = db.supabase.table("artists").select("*").eq("name", artist).execute()
        # if artist_result.data:
        #     existing_genres = artist_result.data[0].get("genres", [])
        #     combined_genres = list(set(existing_genres + enhanced_genres))
        #     
        #     db.supabase.table("artists").update({
        #         "enhanced_genres": enhanced_genres,
        #         "genres": combined_genres
        #     }).eq("id", artist_result.data[0]["id"]).execute()
        #     
        #     print(f"✅ Enhanced genres saved for {artist}")
        
        print("\n💡 This enhanced genre data can be used to:")
        print("  • Improve genre-based matching")
        print("  • Enhance user preference computation")
        print("  • Better music discovery recommendations")
        
    except Exception as e:
        print(f"❌ Genre processing failed: {e}")
else:
    print("⚠️ Skipping enhanced processing - missing database or extractor")

📊 Original tags count: 10
🎵 Filtered genres count: 8
✨ Enhanced genres: ['synthwave', 'alternative rock', 'synthpop', 'alt-pop', 'pop', 'electropop', 'rnb', 'electronic']

💡 This enhanced genre data can be used to:
  • Improve genre-based matching
  • Enhance user preference computation
  • Better music discovery recommendations


In [27]:
def refine_spotify_genres_from_database(db, user_id=None):
    """
    Refine all Spotify genres in the database using Last.fm genre extraction logic
    
    Args:
        db: MusicMatchDB instance
        user_id: Optional - refine genres for specific user, or None for all artists
    
    Returns:
        Dict with refinement results
    """
    
    if not db or not extractor:
        print("❌ Database or extractor not available")
        return {}
    
    try:
        # Get all unique genres from artists in database
        if user_id:
            # Get genres from user's top artists
            result = db.supabase.table("user_top_artists").select(
                "artists!inner(genres)"
            ).eq("user_id", user_id).execute()
            
            all_genres = []
            for item in result.data:
                if item.get("artists") and item["artists"].get("genres"):
                    all_genres.extend(item["artists"]["genres"])
        else:
            # Get all genres from all artists
            result = db.supabase.table("artists").select("genres").execute()
            all_genres = []
            for item in result.data:
                if item.get("genres"):
                    all_genres.extend(item["genres"])
        
        # Remove duplicates and get unique genres
        unique_genres = list(set(all_genres))
        
        print(f"📊 Found {len(unique_genres)} unique Spotify genres")
        print(f"🔍 Sample original genres: {unique_genres[:10]}")
        
        # Apply genre extraction/refinement
        refined_genres = extractor.extract_genres(unique_genres)
        
        print(f"✨ Refined to {len(refined_genres)} clean genres")
        print(f"🎵 Sample refined genres: {refined_genres[:10]}")
        
        # Show the refinement mapping
        print(f"\n📋 GENRE REFINEMENT RESULTS:")
        print(f"Original count: {len(unique_genres)}")
        print(f"Refined count: {len(refined_genres)}")
        print(f"Reduction: {len(unique_genres) - len(refined_genres)} genres")
        
        # Show what was filtered out
        kept_genres = set(refined_genres)
        original_lower = {g.lower() for g in unique_genres}
        refined_lower = {g.lower() for g in refined_genres}
        filtered_out = original_lower - refined_lower
        
        if filtered_out:
            print(f"\n🗑️ Filtered out genres: {list(filtered_out)[:20]}")
        
        return {
            "original_genres": unique_genres,
            "refined_genres": refined_genres,
            "filtered_count": len(unique_genres) - len(refined_genres),
            "refinement_mapping": dict(zip(unique_genres, refined_genres))
        }
        
    except Exception as e:
        print(f"❌ Error refining genres: {e}")
        return {}

def update_artist_genres_in_database(db, refinement_results):
    """
    Update artists in database with refined genres
    
    Args:
        db: MusicMatchDB instance
        refinement_results: Results from refine_spotify_genres_from_database
    """
    
    if not refinement_results:
        print("❌ No refinement results to apply")
        return
    
    try:
        # Get all artists that need genre updates
        artists_result = db.supabase.table("artists").select("id, name, genres").execute()
        
        updated_count = 0
        
        for artist in artists_result.data:
            if not artist.get("genres"):
                continue
                
            # Refine this artist's genres
            original_genres = artist["genres"]
            refined_genres = extractor.extract_genres(original_genres)
            
            # Only update if genres changed
            if set(original_genres) != set(refined_genres):
                # Update artist with refined genres
                db.supabase.table("artists").update({
                    "genres": refined_genres,
                    # Keep original in a backup field if needed
                    # "original_genres": original_genres
                }).eq("id", artist["id"]).execute()
                
                updated_count += 1
                print(f"✅ Updated genres for {artist['name']}")
                print(f"   Before: {original_genres}")
                print(f"   After:  {refined_genres}")
                print()
        
        print(f"🎉 Updated {updated_count} artists with refined genres!")
        
        # Recompute user genre preferences with refined data
        print("\n🔄 Recomputing user genre preferences with refined data...")
        users_result = db.supabase.table("users").select("id").execute()
        
        for user in users_result.data:
            try:
                db.compute_user_genre_preferences(user["id"], "long_term")
                db.compute_user_genre_preferences(user["id"], "medium_term")
                db.compute_user_genre_preferences(user["id"], "short_term")
            except:
                pass  # Skip if user has no top artists
        
        print("✅ User genre preferences updated with refined genres!")
        
    except Exception as e:
        print(f"❌ Error updating database: {e}")

# Test the refinement on your data
if db is not None and extractor is not None:
    print("🔄 Testing genre refinement on your Spotify data...")
    refinement_results = refine_spotify_genres_from_database(db, user_id=None)  # None = all artists
    
    if refinement_results:
        print(f"\n💭 Would you like to apply these refined genres to your database?")
        print(f"   This will update artist genres and recompute user preferences.")
        print(f"   Uncomment the next line to apply:")
        print(f"   # update_artist_genres_in_database(db, refinement_results)")
else:
    print("⚠️ Skipping refinement test - missing database or extractor")

🔄 Testing genre refinement on your Spotify data...
📊 Found 43 unique Spotify genres
🔍 Sample original genres: ['punjabi hip hop', 'folk pop', 'alternative rock', 'bhajan', 'pop', 'desi pop', 'hindi indie', 'rock', 'hardcore hip hop', 'dream pop']
✨ Refined to 32 clean genres
🎵 Sample refined genres: ['punjabi hip hop', 'folk pop', 'alternative rock', 'pop', 'desi pop', 'hindi indie', 'rock', 'hardcore hip hop', 'dream pop', 'punjabi pop']

📋 GENRE REFINEMENT RESULTS:
Original count: 43
Refined count: 32
Reduction: 11 genres

🗑️ Filtered out genres: ['devotional', 'gujarati garba', 'kollywood', 'desi', 'hip hop', 'bhajan', 'bhangra', 'qawwali', 'sufi', 'acoustic pop', 'bollywood', 'ghazal', 'rap']

💭 Would you like to apply these refined genres to your database?
   This will update artist genres and recompute user preferences.
   Uncomment the next line to apply:
   # update_artist_genres_in_database(db, refinement_results)
📊 Found 43 unique Spotify genres
🔍 Sample original genres: ['pu

In [28]:
# Simple test: Show what genre refinement does
if extractor:
    # Example of messy Spotify genres (typical from your artists)
    messy_spotify_genres = [
        "pop", "dance pop", "electropop", "pop rock", "alternative rock", 
        "indie rock", "rock", "hip hop", "rap", "trap", "southern hip hop",
        "melodic rap", "contemporary r&b", "r&b", "neo soul", "urban contemporary",
        "electronic", "house", "deep house", "tech house", "progressive house",
        "singer-songwriter", "indie folk", "folk pop", "acoustic", "country pop"
    ]
    
    print("🎵 BEFORE Refinement:")
    print(f"   Count: {len(messy_spotify_genres)}")
    print(f"   Genres: {messy_spotify_genres}")
    
    # Apply refinement
    refined = extractor.extract_genres(messy_spotify_genres)
    
    print(f"\n✨ AFTER Refinement:")
    print(f"   Count: {len(refined)}")
    print(f"   Genres: {refined}")
    
    print(f"\n📊 RESULTS:")
    print(f"   Removed {len(messy_spotify_genres) - len(refined)} redundant/messy genres")
    print(f"   Kept {len(refined)} clean, standardized genres")
    
    # Show specific changes
    removed = set(messy_spotify_genres) - set(refined)
    if removed:
        print(f"\n🗑️ Removed genres: {list(removed)}")
else:
    print("⚠️ Extractor not available")

🎵 BEFORE Refinement:
   Count: 26
   Genres: ['pop', 'dance pop', 'electropop', 'pop rock', 'alternative rock', 'indie rock', 'rock', 'hip hop', 'rap', 'trap', 'southern hip hop', 'melodic rap', 'contemporary r&b', 'r&b', 'neo soul', 'urban contemporary', 'electronic', 'house', 'deep house', 'tech house', 'progressive house', 'singer-songwriter', 'indie folk', 'folk pop', 'acoustic', 'country pop']

✨ AFTER Refinement:
   Count: 21
   Genres: ['pop', 'electropop', 'post-rock', 'alternative rock', 'indie rock', 'rock', 'hip-hop', 'trap', 'southern hip hop', 'melodic rap', 'contemporary rb', 'rnb', 'neo-soul', 'urban contemporary', 'electronic', 'house', 'tech house', 'progressive house', 'indie folk', 'folk pop', 'country pop']

📊 RESULTS:
   Removed 5 redundant/messy genres
   Kept 21 clean, standardized genres

🗑️ Removed genres: ['pop rock', 'singer-songwriter', 'acoustic', 'hip hop', 'r&b', 'neo soul', 'contemporary r&b', 'dance pop', 'deep house', 'rap']


In [29]:
def refine_specific_artist_genres(db, artist_name_or_id, use_lastfm=True):
    """
    Refine genres for a specific artist using both Spotify data and Last.fm
    
    Args:
        db: MusicMatchDB instance
        artist_name_or_id: Artist name or Spotify ID
        use_lastfm: Whether to enhance with Last.fm data
        
    Returns:
        Dict with original and refined genres
    """
    if not db or not extractor:
        print("❌ Database or extractor not available")
        return {}
    
    try:
        # Find the artist in database
        if len(artist_name_or_id) == 22:  # Spotify ID length
            artist_result = db.supabase.table("artists").select("*").eq("id", artist_name_or_id).execute()
        else:
            artist_result = db.supabase.table("artists").select("*").ilike("name", f"%{artist_name_or_id}%").execute()
        
        if not artist_result.data:
            print(f"❌ Artist '{artist_name_or_id}' not found in database")
            return {}
        
        artist = artist_result.data[0]
        original_genres = artist.get("genres", [])
        
        print(f"🎤 Artist: {artist['name']}")
        print(f"📊 Original Spotify genres: {original_genres}")
        
        # Refine Spotify genres
        refined_spotify = extractor.extract_genres(original_genres) if original_genres else []
        
        # Optionally enhance with Last.fm
        enhanced_genres = refined_spotify.copy()
        
        if use_lastfm and API_KEY:
            try:
                # Get Last.fm tags for this artist
                params = {
                    "method": "artist.gettoptags",
                    "artist": artist['name'],
                    "api_key": API_KEY,
                    "format": "json"
                }
                
                response = requests.get("https://ws.audioscrobbler.com/2.0", params=params)
                data = response.json()
                
                if 'toptags' in data and 'tag' in data['toptags']:
                    lastfm_tags = [tag['name'] for tag in data['toptags']['tag']]
                    lastfm_genres = extractor.extract_genres(lastfm_tags)
                    
                    # Combine and deduplicate
                    combined = list(set(refined_spotify + lastfm_genres))
                    enhanced_genres = extractor.extract_genres(combined)  # Final cleanup
                    
                    print(f"🌐 Last.fm raw tags: {lastfm_tags[:10]}")
                    print(f"🎵 Last.fm refined genres: {lastfm_genres}")
                    
            except Exception as e:
                print(f"⚠️ Last.fm lookup failed: {e}")
        
        print(f"✨ Final refined genres: {enhanced_genres}")
        
        # Show the changes
        added = set(enhanced_genres) - set(original_genres)
        removed = set(original_genres) - set(enhanced_genres)
        
        if added:
            print(f"➕ Added genres: {list(added)}")
        if removed:
            print(f"➖ Removed genres: {list(removed)}")
        
        return {
            "artist_id": artist["id"],
            "artist_name": artist["name"],
            "original_genres": original_genres,
            "refined_spotify": refined_spotify,
            "enhanced_genres": enhanced_genres,
            "added": list(added),
            "removed": list(removed)
        }
        
    except Exception as e:
        print(f"❌ Error refining artist genres: {e}")
        return {}

def refine_user_specific_genres(db, user_id, time_range="long_term", apply_to_db=False):
    """
    Refine genres specifically for a user based on their top artists
    
    Args:
        db: MusicMatchDB instance
        user_id: User's database ID
        time_range: Spotify time range (short_term, medium_term, long_term)
        apply_to_db: Whether to update the database with refined data
        
    Returns:
        Dict with user's refined genre analysis
    """
    if not db or not extractor:
        print("❌ Database or extractor not available")
        return {}
    
    try:
        # Get user's top artists for the specified time range
        user_artists_result = db.supabase.table("user_top_artists").select(
            "artist_id, position, artists!inner(name, genres)"
        ).eq("user_id", user_id).eq("time_range", time_range).order("position").execute()
        
        if not user_artists_result.data:
            print(f"❌ No top artists found for user in {time_range} range")
            return {}
        
        print(f"👤 Analyzing genres for user's top {len(user_artists_result.data)} artists ({time_range})")
        
        # Collect all genres from user's top artists
        all_genres = []
        artist_details = []
        
        for item in user_artists_result.data:
            artist = item["artists"]
            genres = artist.get("genres", [])
            all_genres.extend(genres)
            
            artist_details.append({
                "name": artist["name"],
                "position": item["position"],
                "original_genres": genres,
                "refined_genres": extractor.extract_genres(genres) if genres else []
            })
        
        # Get original genre preferences
        original_prefs = db.supabase.table("user_genre_preferences").select(
            "genre, frequency, weight"
        ).eq("user_id", user_id).eq("time_range", time_range).order("weight", desc=True).execute()
        
        # Refine all genres
        unique_original = list(set(all_genres))
        refined_genres = extractor.extract_genres(unique_original)
        
        print(f"📊 Original genres: {len(unique_original)} → Refined: {len(refined_genres)}")
        print(f"🎵 Top refined genres: {refined_genres[:10]}")
        
        # Calculate new frequency distribution
        refined_freq = {}
        for artist in artist_details:
            for genre in artist["refined_genres"]:
                refined_freq[genre] = refined_freq.get(genre, 0) + 1
        
        # Calculate weights (same logic as database_helper)
        total_artists = len(artist_details)
        refined_preferences = []
        
        for genre, frequency in refined_freq.items():
            weight = frequency / total_artists
            refined_preferences.append({
                "genre": genre,
                "frequency": frequency,
                "weight": round(weight, 2)
            })
        
        refined_preferences.sort(key=lambda x: x["weight"], reverse=True)
        
        print(f"\n🔥 Top 10 refined genre preferences:")
        for pref in refined_preferences[:10]:
            print(f"  • {pref['genre']}: {pref['weight']:.2f} ({pref['frequency']} artists)")
        
        # Apply to database if requested
        if apply_to_db:
            print(f"\n🔄 Updating database with refined genres...")
            
            # Delete old preferences
            db.supabase.table("user_genre_preferences").delete().eq(
                "user_id", user_id
            ).eq("time_range", time_range).execute()
            
            # Insert new refined preferences
            db.supabase.table("user_genre_preferences").insert([
                {
                    "user_id": user_id,
                    "genre": pref["genre"],
                    "frequency": pref["frequency"],
                    "weight": pref["weight"],
                    "time_range": time_range
                }
                for pref in refined_preferences
            ]).execute()
            
            print("✅ Database updated with refined genre preferences!")
        
        return {
            "user_id": user_id,
            "time_range": time_range,
            "total_artists": total_artists,
            "original_genres": unique_original,
            "refined_genres": refined_genres,
            "original_preferences": original_prefs.data,
            "refined_preferences": refined_preferences,
            "artist_details": artist_details
        }
        
    except Exception as e:
        print(f"❌ Error refining user genres: {e}")
        return {}

def analyze_track_genres_from_artists(db, track_id_or_name, enhance_with_lastfm=True):
    """
    Analyze and refine genres for a specific track based on its artists
    
    Args:
        db: MusicMatchDB instance
        track_id_or_name: Track Spotify ID or name
        enhance_with_lastfm: Whether to enhance with Last.fm data
        
    Returns:
        Dict with track genre analysis
    """
    if not db or not extractor:
        print("❌ Database or extractor not available")
        return {}
    
    try:
        # Find the track
        if len(track_id_or_name) == 22:  # Spotify ID
            track_result = db.supabase.table("tracks").select(
                "*, track_artists!inner(artists!inner(name, genres))"
            ).eq("id", track_id_or_name).execute()
        else:
            track_result = db.supabase.table("tracks").select(
                "*, track_artists!inner(artists!inner(name, genres))"
            ).ilike("name", f"%{track_id_or_name}%").execute()
        
        if not track_result.data:
            print(f"❌ Track '{track_id_or_name}' not found")
            return {}
        
        track = track_result.data[0]
        artists = [ta["artists"] for ta in track["track_artists"]]
        
        print(f"🎵 Track: {track['name']}")
        print(f"👥 Artists: {[a['name'] for a in artists]}")
        
        # Collect genres from all artists
        all_genres = []
        artist_genre_details = []
        
        for artist in artists:
            genres = artist.get("genres", [])
            refined = extractor.extract_genres(genres) if genres else []
            
            all_genres.extend(refined)
            artist_genre_details.append({
                "name": artist["name"],
                "original_genres": genres,
                "refined_genres": refined
            })
        
        # Get unique track genres
        track_genres = list(set(all_genres))
        
        print(f"🎭 Track genres from artists: {track_genres}")
        
        # Enhance with Last.fm album data if available
        enhanced_genres = track_genres.copy()
        
        if enhance_with_lastfm and API_KEY and track.get("album_id"):
            try:
                # Try to get album info for Last.fm lookup
                album_result = db.supabase.table("albums").select("name").eq("id", track["album_id"]).execute()
                
                if album_result.data:
                    album_name = album_result.data[0]["name"]
                    artist_name = artists[0]["name"]  # Use first artist
                    
                    params = {
                        "method": "album.gettoptags",
                        "artist": artist_name,
                        "album": album_name,
                        "api_key": API_KEY,
                        "format": "json"
                    }
                    
                    response = requests.get("https://ws.audioscrobbler.com/2.0", params=params)
                    data = response.json()
                    
                    if 'toptags' in data and 'tag' in data['toptags']:
                        lastfm_tags = [tag['name'] for tag in data['toptags']['tag']]
                        lastfm_genres = extractor.extract_genres(lastfm_tags)
                        
                        # Combine and clean
                        combined = list(set(track_genres + lastfm_genres))
                        enhanced_genres = extractor.extract_genres(combined)
                        
                        print(f"🌐 Last.fm album genres: {lastfm_genres}")
                        
            except Exception as e:
                print(f"⚠️ Last.fm enhancement failed: {e}")
        
        print(f"✨ Final track genres: {enhanced_genres}")
        
        return {
            "track_id": track["id"],
            "track_name": track["name"],
            "artists": [a["name"] for a in artists],
            "artist_genre_details": artist_genre_details,
            "track_genres": track_genres,
            "enhanced_genres": enhanced_genres
        }
        
    except Exception as e:
        print(f"❌ Error analyzing track genres: {e}")
        return {}

# Usage examples:
print("🎯 SPECIFIC GENRE REFINEMENT FUNCTIONS LOADED!")
print("\n📋 Available functions:")
print("  1. refine_specific_artist_genres(db, 'Artist Name')")
print("  2. refine_user_specific_genres(db, user_id, time_range='long_term')")
print("  3. analyze_track_genres_from_artists(db, 'Track Name')")
print("\n💡 Examples:")
print("  # Refine specific artist:")
print("  # result = refine_specific_artist_genres(db, 'Drake')")
print("  ")
print("  # Refine user's genres and apply to database:")
print("  # result = refine_user_specific_genres(db, user_id, apply_to_db=True)")
print("  ")
print("  # Analyze track genres:")
print("  # result = analyze_track_genres_from_artists(db, 'Blinding Lights')")

🎯 SPECIFIC GENRE REFINEMENT FUNCTIONS LOADED!

📋 Available functions:
  1. refine_specific_artist_genres(db, 'Artist Name')
  2. refine_user_specific_genres(db, user_id, time_range='long_term')
  3. analyze_track_genres_from_artists(db, 'Track Name')

💡 Examples:
  # Refine specific artist:
  # result = refine_specific_artist_genres(db, 'Drake')
  
  # Refine user's genres and apply to database:
  # result = refine_user_specific_genres(db, user_id, apply_to_db=True)
  
  # Analyze track genres:
  # result = analyze_track_genres_from_artists(db, 'Blinding Lights')


In [30]:
# 🧪 TEST SPECIFIC REFINEMENT FUNCTIONS

if db is not None and extractor is not None:
    print("🎯 Testing specific genre refinement functions...")
    
    # First, let's see what artists we have in the database
    sample_artists = db.supabase.table("artists").select("name, genres").limit(5).execute()
    
    if sample_artists.data:
        print(f"\n📋 Sample artists in your database:")
        for i, artist in enumerate(sample_artists.data, 1):
            print(f"  {i}. {artist['name']} - {len(artist.get('genres', []))} genres")
        
        # Test with the first artist
        test_artist = sample_artists.data[0]['name']
        print(f"\n🧪 Testing refinement for: {test_artist}")
        
        # Refine specific artist
        artist_result = refine_specific_artist_genres(db, test_artist, use_lastfm=True)
        
        if artist_result:
            print(f"\n📊 ARTIST REFINEMENT RESULTS:")
            print(f"   Original: {len(artist_result['original_genres'])} genres")
            print(f"   Enhanced: {len(artist_result['enhanced_genres'])} genres")
            if artist_result['added']:
                print(f"   Added: {artist_result['added']}")
            if artist_result['removed']:
                print(f"   Removed: {artist_result['removed']}")
    
    # Check if we have any users to test user-specific refinement
    users = db.supabase.table("users").select("id").limit(1).execute()
    
    if users.data:
        test_user = users.data[0]['id']
        print(f"\n👤 Testing user-specific refinement for user: {test_user[:8]}...")
        
        user_result = refine_user_specific_genres(db, test_user, time_range="long_term", apply_to_db=False)
        
        if user_result:
            print(f"\n📊 USER REFINEMENT RESULTS:")
            print(f"   Total artists analyzed: {user_result['total_artists']}")
            print(f"   Original unique genres: {len(user_result['original_genres'])}")
            print(f"   Refined unique genres: {len(user_result['refined_genres'])}")
            print(f"   Top 5 refined preferences:")
            for pref in user_result['refined_preferences'][:5]:
                print(f"     • {pref['genre']}: {pref['weight']:.2f}")
    
    # Test track analysis
    sample_tracks = db.supabase.table("tracks").select("name").limit(3).execute()
    
    if sample_tracks.data:
        test_track = sample_tracks.data[0]['name']
        print(f"\n🎵 Testing track genre analysis for: {test_track}")
        
        track_result = analyze_track_genres_from_artists(db, test_track, enhance_with_lastfm=True)
        
        if track_result:
            print(f"\n📊 TRACK ANALYSIS RESULTS:")
            print(f"   Artists: {', '.join(track_result['artists'])}")
            print(f"   Track genres: {track_result['track_genres']}")
            print(f"   Enhanced genres: {track_result['enhanced_genres']}")

else:
    print("⚠️ Database or extractor not available for testing")

🎯 Testing specific genre refinement functions...

📋 Sample artists in your database:
  1. AP Dhillon - 5 genres
  2. Saransh Peer - 0 genres
  3. Pritam - 3 genres
  4. Joji - 0 genres
  5. OneRepublic - 1 genres

🧪 Testing refinement for: AP Dhillon
🎤 Artist: AP Dhillon
📊 Original Spotify genres: ['punjabi hip hop', 'punjabi pop', 'desi', 'bhangra', 'desi hip hop']
🎤 Artist: AP Dhillon
📊 Original Spotify genres: ['punjabi hip hop', 'punjabi pop', 'desi', 'bhangra', 'desi hip hop']
🌐 Last.fm raw tags: ['Bhangra', 'canada', 'india', 'Indian', 'Punjabi', 'punjab', 'Indian roots', 'Punjabi Trap', 'desi trap']
🎵 Last.fm refined genres: ['indie']
✨ Final refined genres: ['punjabi hip hop', 'indie', 'desi hip hop']
➕ Added genres: ['indie']
➖ Removed genres: ['punjabi pop', 'bhangra', 'desi']

📊 ARTIST REFINEMENT RESULTS:
   Original: 5 genres
   Enhanced: 3 genres
   Added: ['indie']
   Removed: ['punjabi pop', 'bhangra', 'desi']

👤 Testing user-specific refinement for user: a4122524...
🌐 L